# 43008: Reinforcement Learning - WOW Naval Fleet: MAPPO vs Double DQN

## Stage 5: Archipelago

[Imitation](StageP-Imitation.ipynb) · [Stage 0](Stage0-Gunnery.ipynb) · [Stage 1](Stage1-Capture.ipynb) · [Stage 2](Stage2-Defend.ipynb) · [Stage 3](Stage3-Duel.ipynb) · [Stage 4](Stage4-King-of-the-Hill.ipynb) · **Stage 5** · [Stage 6](Stage6-Domination.ipynb) · [Stage 7](Stage7-Open.ipynb) · [Stage 8](Stage8-Archipelago-9v9.ipynb) · [the full walkthrough](WOW-MAPPO-Simple.ipynb)

| Stage | Battle | Fleet | Opponent | Promotion requirement |
|---|---|---|---|---|
| 5 - Archipelago | `archipelago_3v3` | 3v3 mixed fleet | Veteran rule AI 60% · self-play 40% (itself, champion, past selves, the other method) | won in ≥70% of the last 150 battles against the rule AI |

### What the fleet learns here

The same three ships a side, now among islands with **three capture circles** (A, B, C) and 8 minutes. This is where the commander earns its place: **which ship holds which circle, and who hunts**. Islands break line of sight, so spotting, smoke, radar and cover matter. Team spirit 0.5.

Measured earlier: imitation alone (naval_rl) won 45%, flat MAPPO 27%.

### The ships

| Class | Our fleet | Enemy fleet |
|---|---|---|
| Battleship (BB) | 1 | 1 |
| Cruiser (CA) | 1 | 1 |
| Destroyer (DD) | 1 | 1 |
| Submarine (SS) | 0 | 0 |
| **Total** | **3** | **3** |

| Class | | What it does |
|---|---|---|
| **Battleship** | BB | the biggest guns and the thickest armour; slow, long range - wins duels, anchors the line |
| **Cruiser** | CA | medium guns that fire fast, radar / hydro / smoke support; the all-rounder |
| **Destroyer** | DD | the fastest, torpedoes and smoke, hardest to spot - the circle capper and scout |
| **Submarine** | SS | dives under water, torpedoes, very hard to spot or hit while submerged |

### One model per method, through every stage

The stage notebooks train **two runs** - the MAPPO fleet (`RUN_NAME`) and the Double DQN fleet (`RUN_NAME + "_ddqn"`) - each through every stage. A session continues each run's latest checkpoint, which carries everything learned so far (networks, optimisers, the league, the timestep count, the stage) and its battle log, from which the promotion window continues where the last session stopped.

* **Training is continuous.** Not passed yet? **Run this notebook again** - each session continues the last one (yours or a teammate's), from `models/model_latest.pt`, refreshed every `save_interval`. `KEEP_GOING = True` carries on into the next stages in the same session instead of stopping at the promotion.
* `TRAIN_MAPPO` / `TRAIN_DDQN` (configuration cell) switch either method's training on or off for this session. A run that is on another stage is not trained here, only evaluated for the comparison.
* Passed? That run stops and saves, and moves to stage 6: open [Stage 6](Stage6-Domination.ipynb).


### Two methods, the same fleet

Every stage notebook trains **two** runs on the same battles - one per method - and compares them:

| | **MAPPO** (`simple_mappo/mappo.py`) | **Double DQN** (`simple_mappo/ddqn.py`) |
|---|---|---|
| Family | policy gradient (PPO, Week 10) | value-based Q-learning (DQN, Week 9) |
| What the captain's head scores are | logits: sample an option | Q-values (dueling: V + A): take the best option, epsilon-greedy while training |
| Learns from | the battles it just played (on-policy) | a replay buffer of chunks of past battles (off-policy, R2D2's sequence replay) |
| Target | GAE advantages from a centralised critic | Double Q-learning: the online network picks the next action, the target network scores it; n-step returns |
| Six action heads | one probability per head | one Q per head, every head trained to the same target (branching DQN, BDQ) |
| Fleet commander | PPO at a 10 s clock | Q-learning on (ship, order) at a 10 s clock (h-DQN, intra-option) |
| Many ships | shared weights, centralised critic | shared weights, each ship on its own reward (`mixer="qmix"` for QMIX, the StarCraft method) |
| Shared | the captain network (transformer + GRU + pointer heads), the commander, the rewards, the curriculum, self-play, the imitation start | ← same |

Each run climbs the curriculum on its own (`runs_simple/fleet` and `runs_simple/fleet_ddqn`), so a stage notebook only trains a run that is on its stage; the other one is still evaluated here, for the comparison.

### MAPPO in one picture

| Piece | What it does | Where it comes from | Course link |
|---|---|---|---|
| **Captain** (every ship, one shared network) | its own view → 6 action heads; a transformer over ships/circles/islands, a GRU memory, pointer heads for *which target* and *which circle* | AlphaStar, OpenAI Five | W10-C custom feature extractor; frame stacking → memory |
| **Fleet commander** | every 10 s: an order per ship - free / engage / hold circle k | Honor of Kings' hierarchical macro strategy | the same PPO at a slower clock |
| **Centralised critic** | sees the true battle (training only); one value per reward group + win probability | MAPPO, Honor of Kings multi-head value | W10-A/B actor-critic |
| **Update** | PPO with dual clip, value normalisation and clipping | MAPPO, Honor of Kings | W10-B PPO |
| **Warm start** | the captain first copies the Elite rule AI (StageP), then a fading KL penalty keeps it close | AlphaStar | W9-A supervised training loop |
| **Self-play & league** | from stage 3: mirror battles (the fleet on **both** sides, **both** learning), its **champion** (replaced only when beaten in 55% of 40 battles), past copies (PFSP), and the **other method's fleet** (cross-play), with Elo ratings; the rule AI stays as the anchor promotion is judged against | OpenAI Five, AlphaGo Zero, AlphaStar | W9-C/D frozen target network |
| **Curriculum** | 8 stages, each passed at ≥70% of its own goal; earlier stages are replayed 10-15% of the time | TiZero, OpenAI Five | - |

## Setup

On SageMaker, run this from the unzipped bundle (`<folder>/Training/`). On your computer, open it from the repository's `Training/` folder. Either way the battles are the **real Unity game** (the headless player in `Builds/NavalTrainer`, built from this project's `Assets/`), not a simplified copy.

In [ ]:
import os, time, zipfile
# SageMaker Studio (/home/sagemaker-user) or a SageMaker notebook instance (/home/ec2-user/SageMaker)
ON_SAGEMAKER = any(os.path.isdir(p) for p in ("/home/sagemaker-user", "/home/ec2-user/SageMaker"))
home = os.path.expanduser("~/SageMaker") if os.path.isdir(os.path.expanduser("~/SageMaker")) else os.path.expanduser("~")
bundle = os.path.join(home, "wow_sagemaker.zip")
if ON_SAGEMAKER and os.path.basename(os.getcwd()) != "Training":
    if not os.path.isdir(os.path.join(home, "wow")):
        assert os.path.exists(bundle), f"upload wow_sagemaker.zip to {home} first"
        zipfile.ZipFile(bundle).extractall(os.path.join(home, "wow"))
    %cd {home}/wow/Training

# Uploading a new wow_sagemaker.zip does not replace the unzipped folder: check which version runs here
running = open("BUNDLE_VERSION").read().strip() if os.path.exists("BUNDLE_VERSION") else "an older one"
if ON_SAGEMAKER and os.path.exists(bundle):
    with zipfile.ZipFile(bundle) as z:
        uploaded = z.read("Training/BUNDLE_VERSION").decode().strip() if "Training/BUNDLE_VERSION" in z.namelist() else "unknown"
    if uploaded != running:
        print(f"*** {bundle} is version {uploaded}, but this notebook runs version {running}. Unzip it into a new folder "
              f"and open this notebook from there. ***")
print(os.getcwd(), "| bundle", running, "| glibc", os.confstr("CS_GNU_LIBC_VERSION"))

In [ ]:
# install only what is missing (PyTorch as the CPU build)
import importlib.util
missing = [pkg for mod, pkg in (("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "pillow"))
           if importlib.util.find_spec(mod) is None]
if missing:
    !pip install -q {" ".join(missing)}
if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import os
from collections import OrderedDict

import numpy as np
import pandas as pd

from simple_mappo import (NavalEnv, MAPPO, FleetDDQN, Monitor, CurriculumCallback, SaveOnIntervalCallback,
                          BestModelCallback, VideoCallback, TeamStorage, TeamSyncCallback, DEFAULT_CURRICULUM,
                          open_run, training_status, evaluate, plot_results, plot_curriculum, plot_reward_terms,
                          plot_progress, plot_comparison, compare_stages, show_videos)

## Team checkpoints

Use the **same `RUN_NAME` in every notebook** (and as your teammates): that is the one line of training the stages hand over along - one per method. Only one person can train a run at a time.

In [ ]:
YOUR_NAME = "your-name"              # shown to your teammates
RUN_NAME = "fleet"                   # the MAPPO run - the same in every stage notebook (and StageP)
if ON_SAGEMAKER:
    from simple_mappo.aws import team_storage_uri
    TEAM_STORAGE = team_storage_uri()     # s3://sagemaker-<region>-<account>/wow-mappo, or "s3://<team-bucket>/wow-mappo"
else:
    TEAM_STORAGE = None              # e.g. "s3://<team-bucket>/wow-mappo"; None trains on this machine only

storage = TeamStorage(TEAM_STORAGE, RUN_NAME, user=YOUR_NAME) if TEAM_STORAGE else None
run_dir = f"runs_simple/{RUN_NAME}"
log_dir, models_dir, video_dir = (os.path.join(run_dir, d) for d in ("logs", "models", "videos"))
os.makedirs(models_dir, exist_ok=True)
if storage:
    storage.show_status()

In [ ]:
DDQN_RUN_NAME = RUN_NAME + "_ddqn"   # the Double DQN run: its own checkpoints, logs and curriculum stage
dqn_storage = TeamStorage(TEAM_STORAGE, DDQN_RUN_NAME, user=YOUR_NAME) if TEAM_STORAGE else None
dqn_run_dir = f"runs_simple/{DDQN_RUN_NAME}"
dqn_log_dir, dqn_models_dir, dqn_video_dir = (os.path.join(dqn_run_dir, d) for d in ("logs", "models", "videos"))
os.makedirs(dqn_models_dir, exist_ok=True)
if dqn_storage:
    dqn_storage.show_status()

## Configuration

Each method's settings are **identical in every notebook**. They only shape a brand-new model; a continued model keeps the settings it was created with. `n_timesteps` is this session's training, per method; rerun the notebook to continue.

In [ ]:
STAGE = 5
FORCE_STAGE = False                  # True: train this stage even if a run is on another one
KEEP_GOING = False                   # True: when this stage is passed, carry on into the next stages
                                     # in this same session instead of stopping (continuous training)
TRAIN_MAPPO = True                   # train the MAPPO run in this session
TRAIN_DDQN = True                    # train the Double DQN run in this session

config = OrderedDict([
    ('n_envs', max(1, min(16, os.cpu_count() // 2))),     # battles in parallel, one Unity process each
    ('decision_period', 1.0),          # game seconds between decisions
    ('sim_dt', 0.08),                  # game seconds per simulated frame (fastest stable)
    ('n_timesteps', 500_000),          # this session's training, per method
])
mappo_settings = dict(learning_rate=3e-4, critic_learning_rate=5e-4, n_steps=256, n_minibatches=4, n_epochs=5,
                      gamma=0.995, gae_lambda=0.95, clip_range=0.2, dual_clip=3.0, ent_coef=0.01,
                      d_model=64, attn_layers=2, hidden_size=128, recurrent=True,      # the networks
                      commander=True, commander_period=10,                             # the fleet commander
                      kl_bc_coef=0.1, opponent_refresh=10)                             # imitation anchor, league
ddqn_settings = dict(learning_rate=2.5e-4, buffer_size=40_000, learning_starts=4_000, batch_size=32,
                     chunk_len=32, gradient_steps=4, gamma=0.99, n_step=5, target_update_interval=250,
                     exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_timesteps=300_000,
                     kl_bc_coef=0.5,                  # kickstarting: the pull towards the imitation clone, fading
                     mixer="none",                    # "qmix": one fleet value, mixed by hypernetworks (SMAC)
                     d_model=64, attn_layers=2, hidden_size=128, recurrent=True,       # the same networks as MAPPO
                     commander=True, commander_period=10)                             # the same commander
save_interval = config['n_timesteps'] // 3

pd.Series(DEFAULT_CURRICULUM[STAGE], name=f"stage {STAGE}").to_frame()

## Environment

Launches `n_envs` headless Unity players - the real game. Both methods fight on the same players, each with its own battle log (`Monitor`) and its own curriculum stage. The observation has the same size in every stage (room for 9 ships a side and 5 circles), which is what lets one network play them all.

In [ ]:
battles = NavalEnv(stages="default", start_stage=STAGE, n_envs=config['n_envs'],
                   decision_period=config['decision_period'], sim_dt=config['sim_dt'],
                   unity_log_dir=os.path.join(run_dir, "unity_logs"))
env = Monitor(battles, log_dir)              # the MAPPO run's battles
env_dqn = Monitor(battles, dqn_log_dir)      # the Double DQN run's battles
print("battles in parallel:", battles.n_envs, "| token sizes", battles.dims)

## The two models

`open_run` loads each run's latest checkpoint with all its training memory. It says when a run is on another stage: that run is then only evaluated here.

In [ ]:
imitation = os.path.join(models_dir, "imitation.pt")
model, mappo_here = open_run(env, STAGE, run_dir, log_dir, "mappo", storage=storage, force=FORCE_STAGE, **mappo_settings)
dqn, dqn_here = open_run(env_dqn, STAGE, dqn_run_dir, dqn_log_dir, "ddqn", storage=dqn_storage, force=FORCE_STAGE,
                         imitation_path=imitation, **ddqn_settings)
TRAIN_MAPPO, TRAIN_DDQN = TRAIN_MAPPO and mappo_here, TRAIN_DDQN and dqn_here
if storage and TRAIN_MAPPO:
    storage.lock()                   # stops here if a teammate is training this run right now
if dqn_storage and TRAIN_DDQN:
    dqn_storage.lock()
print(f"MAPPO: {'trains here' if TRAIN_MAPPO else 'evaluated only'} | Double DQN: {'trains here' if TRAIN_DDQN else 'evaluated only'}")
model.actor if model else None       # the captain (and the commander), as in the Week 10 notebooks

## Baselines

What the trained fleets have to beat on this stage: random legal actions, and a simple hand-written tactic. `pass_rate` is the stage's own goal (see the table at the top). Set `RUN_BASELINES = False` to skip them in later sessions.

In [ ]:
class Script:
    """Every ship sails to the first capture circle at full speed and fires at will."""
    def predict(self, obs, deterministic=False):
        E, N = obs["alive"].shape
        actions = np.zeros((E, N, len(env.head_sizes)), dtype=np.int32)
        actions[..., 0] = np.where(obs["action_mask"][..., 15] > 0.5, 15, 0)   # move option 15 = sail to circle 0
        return actions
SCRIPT_NAME = "scripted: go to the circle, fire at will"

RUN_BASELINES = True
baselines = {}
if RUN_BASELINES:
    baselines = {"random": evaluate(None, env, n_battles=24, stage=STAGE),
                 SCRIPT_NAME: evaluate(Script(), env, n_battles=24, stage=STAGE)}
    display(pd.DataFrame(baselines).T[["stage", "battles", "win_rate", "pass_rate", "mean_reward"]])

## Train MAPPO

Training stops by itself as soon as the promotion requirement is met (unless `KEEP_GOING`), or after `n_timesteps`. `models/model_best.pt` always holds the checkpoint with the best rolling pass rate. The team folder gets a status update every 10,000 timesteps and a checkpoint every `save_interval`.

**Replays**: five times a session a battle is recorded and **shown right here, under the training table**, while training goes on (`runs_simple/<RUN_NAME>/videos/`, `.gif` on SageMaker - double-click one in the file browser).

In the table: `win_rate` / `pass_rate` against the stage's own opponent, `win_rate_self_play` in battles against itself (about 0.5 in mirror battles - both sides are the fleet), `elo` the fleets rating, `orders_free` the share of the commanders orders that leave the ship free.

In [ ]:
def callbacks_for(run_model, run_dir, models_dir, video_dir, run_storage):
    curriculum = CurriculumCallback(stop_on_promotion=not KEEP_GOING)
    callbacks = [SaveOnIntervalCallback(save_interval, models_dir),     # also refreshes model_latest.pt
                 curriculum,
                 BestModelCallback(models_dir),
                 VideoCallback(every=config['n_timesteps'] // 5, video_dir=video_dir, show=True)]   # replays appear below
    if run_storage:
        callbacks.append(TeamSyncCallback(run_storage, run_dir, push_every=save_interval, status_every=10_000))
    return callbacks, curriculum

if TRAIN_MAPPO:
    model.set_rival(dqn)             # cross-play: the Double DQN fleet joins MAPPO's league as "rival"
    callbacks, curriculum = callbacks_for(model, run_dir, models_dir, video_dir, storage)
    model.learn(total_timesteps=model.num_timesteps + config['n_timesteps'], callback=callbacks, log_interval=5)
    model.save(os.path.join(models_dir, "model_final"))
    model.save(os.path.join(models_dir, "model_latest"))

## Train Double DQN

The same battles, the same rewards, the same stopping rule - learned with Double DQN. The first `learning_starts` decisions only fill the replay buffer (it is not saved with the model, so every session refills it first); then every 32 decisions per battle bring `gradient_steps` updates on chunks sampled from it.

Cross-play: before each method trains, the other one's fleet - its champion, if it has one - joins its league as the **rival** (`set_rival`), so each learns to beat a differently trained fleet too. A *new champion* message means the fleet just beat its best self.

In the table: `q_loss` the TD error, `q_value` / `target_value` the Q-values and their targets (both should rise as the fleet starts winning), `epsilon` the exploration rate, `replay_decisions` how full the buffer is.

In [ ]:
if TRAIN_DDQN:
    dqn.set_rival(model)             # ... and the MAPPO fleet (just trained) joins Double DQN's
    callbacks, dqn_curriculum = callbacks_for(dqn, dqn_run_dir, dqn_models_dir, dqn_video_dir, dqn_storage)
    dqn.learn(total_timesteps=dqn.num_timesteps + config['n_timesteps'], callback=callbacks, log_interval=8)
    dqn.save(os.path.join(dqn_models_dir, "model_final"))
    dqn.save(os.path.join(dqn_models_dir, "model_latest"))

## Save and hand over

Pushes both runs to the team. **If you stop training early (Kernel > Interrupt), run this cell too** (and `model.save(...)` / `dqn.save(...)` from the cells above), so your progress is kept and the runs are free for the next person.

In [ ]:
for name, m, m_env, m_storage, m_run_dir, trained in (("MAPPO", model, env, storage, run_dir, TRAIN_MAPPO),
                                                      ("Double DQN", dqn, env_dqn, dqn_storage, dqn_run_dir, TRAIN_DDQN)):
    if m is None or not trained:
        continue
    if m_storage:
        m_storage.push(m_run_dir, m, status=training_status(m, m_run_dir))
        m_storage.release()
    if m_env.stage > STAGE:
        print(f"{name}: stage {STAGE} passed at {m.num_timesteps:,} timesteps - now on '{m_env.stages[m_env.stage]['name']}': "
              f"open the Stage{m_env.stage} notebook.")
    else:
        print(f"{name}: stage {STAGE} {'- the stopping rule decides when it is done' if STAGE == len(m_env.stages) - 1 else 'not passed yet'} "
              f"({m.num_timesteps:,} timesteps) - run this notebook again to continue.")

## Progress on this stage

The two methods side by side, against the timesteps each has spent **on this stage**: the pass rate (what promotion counts) and the reward.

In [ ]:
if storage:
    storage.show_status()
plot_comparison({"MAPPO": log_dir, "Double DQN": dqn_log_dir}, stage=STAGE)

In [ ]:
plot_curriculum(log_dir)                     # MAPPO: pass rate per stage, one colour each
plot_curriculum(dqn_log_dir)                 # Double DQN
plot_reward_terms(log_dir, stage=STAGE)      # what the MAPPO fleet is being paid for on this stage
plot_reward_terms(dqn_log_dir, stage=STAGE)  # ... and the Double DQN fleet

In [ ]:
plot_progress(log_dir)                       # PPO's diagnostics
plot_progress(dqn_log_dir)                   # Double DQN's diagnostics

In [ ]:
# the commanders: how often each leaves a ship free (1.0 = never gives orders)
for name, d in (("MAPPO", log_dir), ("Double DQN", dqn_log_dir)):
    path = os.path.join(d, "progress.csv")
    if os.path.exists(path):
        p = pd.read_csv(path)
        print(name)
        display(p[p.stage == STAGE][["total_timesteps", "orders_free", "win_rate", "pass_rate"]].tail(5))

In [ ]:
# the leagues: Elo ratings of each fleet, its champion, past copies, the rival (the other method) and the
# rule AI (chess-style; 400 points = 10:1 odds)
for name, m in (("MAPPO", model), ("Double DQN", dqn)):
    if m is not None:
        print(name)
        display(pd.DataFrame(m.league.table(), columns=["player", "elo", "our record against it"]).head(8))

## Evaluation: MAPPO vs Double DQN

Fresh battles on this stage against its own opponent, for every model: MAPPO samples its actions (its natural policy); Double DQN takes its best action (2% random). The best checkpoints are evaluated too.

In [ ]:
results = dict(baselines)
for name, cls, m, mdir in (("MAPPO", MAPPO, model, models_dir), ("Double DQN", FleetDDQN, dqn, dqn_models_dir)):
    if m is None:
        print(f"{name}: no model yet - train it from Stage 0")
        continue
    m_env = env if cls is MAPPO else env_dqn
    results[f"{name} (latest)"] = evaluate(m, m_env, n_battles=40, stage=STAGE)
    best_path = os.path.join(mdir, "model_best.pt")
    if os.path.exists(best_path):
        stage_now = m_env.stage
        best = cls.load(best_path, env=m_env)          # (loading moves that run's stage to the one it was saved on)
        results[f"{name} (best)"] = evaluate(best, m_env, n_battles=40, stage=STAGE)
        m_env.set_stage(stage_now)
columns = ["battles", "win_rate", "pass_rate", "win_rate_stderr", "mean_reward", "enemy_ships_sunk", "own_ships_lost",
           "points", "enemy_points"]
table = pd.DataFrame(results).T
table[[c for c in columns if c in table]]

## Every stage so far

How each method did on every stage of the curriculum, from its battle log: the timesteps it spent there until it passed, its pass rate over its last 50 qualifying battles there, and whether it passed.

In [ ]:
compare_stages({"MAPPO": log_dir, "Double DQN": dqn_log_dir}, env.stages)

## Put the fleets into your Unity game

Exports each run's best checkpoint (or its latest, if that has reached a later stage) as a policy file the game runs in plain C# (`Assets/Scripts/RL/RLPolicy.cs`): MAPPO's as `naval_policy.bin`, Double DQN's as `naval_policy_ddqn.bin`. From the repository they go straight into `Assets/StreamingAssets/RL/`; elsewhere (SageMaker) into the run folders - download them (right-click > Download) and copy them there.

Then in Unity: **Play** > **TRAINING STAGES**; at the bottom pick **TRAINED FLEET: MAPPO** or **DOUBLE DQN**, and **WATCH AI** on any stage (or on the setup screen: **YOUR FLEET: LEARNED** / **ENEMY AI: LEARNED**). From a computer later: `python export_simple.py` and `python export_simple.py --run-name fleet_ddqn`.

In [ ]:
from simple_mappo.export import GAME_POLICY, GAME_POLICY_DDQN, export_policy, pick_checkpoint
in_repo = os.path.isdir(os.path.join("..", "Assets", "StreamingAssets"))
for mdir, rdir, game_file in ((models_dir, run_dir, GAME_POLICY), (dqn_models_dir, dqn_run_dir, GAME_POLICY_DDQN)):
    checkpoint = pick_checkpoint(mdir)        # model_best.pt, or model_latest.pt if it reached a later stage
    if checkpoint is None or checkpoint.endswith("imitation.pt") and mdir == dqn_models_dir:
        continue
    target = game_file if in_repo else os.path.join(rdir, os.path.basename(game_file))
    print("exported", checkpoint, "->", os.path.abspath(export_policy(checkpoint, target)))

## Replays from this stage

The newest replays recorded while training on this stage, MAPPO's then Double DQN's. They are a 2D map drawn from the game's data (the training player has no graphics); to see a fleet in the real game's graphics, export it (above) and use **WATCH AI** in Unity.

In [ ]:
show_videos(video_dir, contains=f"_stage{STAGE}", newest=2)
show_videos(dqn_video_dir, contains=f"_stage{STAGE}", newest=2)

In [ ]:
battles.close()